# 04 – Probabilistic forecast (PINN)

Forecast of the epidemic with the posterior samples estimated in notebook 03. At every week the last 1000 samples of the chain (R0, Delta, tau0) are propagated through the pretrained PINN: every sample gives one trajectory, and the trajectories are summarised by their quantiles. One notebook for the five cases of the paper, chosen with `CASE` as in notebook 03 (**use the same `CASE`, `R0_TRUE`, `NOISE` and `CASE_OVERRIDES`**, they select the folder with the chains):

| `CASE` | Forecasted quantity |
|---|---|
| `"synthetic"` (R0 = 1.5 / 2.0, noise `"light"` / `"mild"`) | new infections |
| `"real"` | new deaths |

`FORECAST_HORIZONS` are the horizons in weeks: 4 for the short-term and 10 for the long-term forecast. The forecast is repeated for every seed in `SEEDS`: 100 independent runs, or only one if `SEEDS = [10]`. With `RESUME = True` the forecasts already done are skipped.

The forecasts are made at the same weeks where the parameters were estimated in notebook 03 (`case.t0_estimation()`): the rolling windows and, for the synthetic scenarios, the extra window at the last week (t0 = 33 for R0 = 1.5, t0 = 23 for R0 = 2.0), whose fit is used in notebook 08 for the retrospective fit and the age sanity check.

For the synthetic cases it is also possible to forecast the trajectory of every age class (`AGE_FORECASTS = True`), with the same posterior samples (no new estimation).

Results: `outputs/posteriors/pinn/<case>/seed_<seed>/forecast_h<horizon>/forecast_results.pickle`. For every week it contains the samples (`raw`), the 50% and 90% intervals (`summary_50`, `summary_90`), the quantiles (`quantiles`), the observations of the window (`y_obs`) and the week (`t0`, `date`). The best seed is chosen afterwards, with the score of the fit, in notebook 10.

No plots here.

**Requires notebook 03.** It reads the chains written by notebook 03 in `outputs/posteriors/pinn/<case>/seed_<seed>/` (same `CASE`, `CASE_OVERRIDES` and `SEEDS`). To run it without repeating the estimation, copy the shipped run `precomputed/posteriors/pinn/<case>/best_WIS/` to `outputs/posteriors/pinn/<case>/seed_010/` and keep `SEEDS = [10]`. The copy also contains the forecasts already computed: with `RESUME = True` they are skipped, so delete the `forecast_*` folders of the copy (or set `RESUME = False`) to compute them again.

In [ ]:
import os

import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm

from pinnvsmech.constants import N_age
from pinnvsmech.data.synthetic import load_age_incidence
from pinnvsmech.forecasting.forecast import run_probabilistic_forecast
from pinnvsmech.inference.cases import load_case
from pinnvsmech.pinn.NN_free_params import NN
from pinnvsmech.paths import PRECOMPUTED_DIR, OUTPUTS_DIR

torch.set_default_dtype(torch.float64)

## Settings

In [ ]:
CASE = "synthetic"                 # "synthetic" or "real"
R0_TRUE = 2.0                      # synthetic only: 1.5 or 2.0
NOISE = "light"                    # synthetic only: "light" or "mild"
CASE_OVERRIDES = {}                # as in notebook 03

SEEDS = [10]                       # one run (seed 10)
# SEEDS = list(range(100))         # 100 independent runs (seeds 0..99)
FORECAST_HORIZONS = [4]            # weeks ahead: 4 (short-term), 10 (long-term) or both [4, 10]
FIT = True                         # True: the forecast includes also the weeks of the window of observations
RESUME = True                      # skip the forecasts already done

AGE_FORECASTS = False              # synthetic only: forecast also every age class

SAMPLE_STRATEGY = "tail"           # posterior samples: last NSAMPLE iterations of the chain
NSAMPLE = 1000
NSIM = 10000                       # MCMC iterations used in notebook 03

PINN_WEIGHTS = PRECOMPUTED_DIR / "pretrained_pinn" / "pretrained_pinn.pth"
# After a new training (notebook 02): OUTPUTS_DIR / "pinn" / "models" / "<RUN_NAME>.pth"
CHAINS_DIR = OUTPUTS_DIR / "posteriors" / "pinn"

## Data and PINN

In [ ]:
case = load_case(CASE, R0=R0_TRUE, noise=NOISE, **CASE_OVERRIDES)
print(case.label)
print(case.settings())

if AGE_FORECASTS and case.kind != "synthetic":
    raise ValueError("AGE_FORECASTS is available only for the synthetic cases (true incidence of the age classes)")

In [ ]:
model = NN(activation=nn.Tanh, dimensions=[3, 100, 100, 100, 100, 100, 6 * len(N_age)])
model.load_state_dict(torch.load(PINN_WEIGHTS, weights_only=True))
model.eval()

## Forecast

In [ ]:
case_dir = CHAINS_DIR / case.label
forecast_args = dict(
    dates=case.dates, dt_train=case.dt_train, pinn_model=model, t0_range=case.t0_estimation(),
    delta_fixed=case.delta_fixed, observable=case.observable, fit=FIT,
    sample_strategy=SAMPLE_STRATEGY, nsample=NSAMPLE, nsim=NSIM,
)

if AGE_FORECASTS:
    # True (noiseless) incidence of every age class, to be compared with the forecast of that class
    _, age_incidence = load_age_incidence(R0_TRUE)
    age_incidence = np.asarray(age_incidence)

for seed in tqdm(SEEDS, desc=case.label):
    seed_dir = case_dir / f"seed_{seed:03d}"
    if not (seed_dir / "mcmc_distributions.pkl").exists():
        raise FileNotFoundError(f"No chains in {seed_dir}: run notebook 03 first (same CASE, seed {seed})")

    for horizon in FORECAST_HORIZONS:
        if RESUME and (seed_dir / f"forecast_h{horizon}" / "forecast_results.pickle").exists():
            pass
        else:
            # Aggregate forecast
            run_probabilistic_forecast(
                chains_dir=seed_dir, obs=case.obs, horizon=horizon, N_age=N_age, **forecast_args,
            )

        if AGE_FORECASTS:
            # One age class at a time: the population of the other classes is set to zero
            for a in range(len(N_age)):
                if RESUME and (seed_dir / f"forecast_age{a}" / "forecast_results.pickle").exists():
                    continue
                N_age_masked = np.zeros_like(N_age)
                N_age_masked[a] = N_age[a]
                run_probabilistic_forecast(
                    chains_dir=seed_dir, obs=age_incidence[:, a], horizon=horizon, N_age=N_age_masked,
                    tag=f"age{a}", **forecast_args,
                )